# Red neuronal para clasificar la calidad de instancias de vino
Implementación del modelo de red neuronal con mejor rendimiento visto en el notebook "[clasificación multiclase](https://www.kaggle.com/code/criser2013/clasificaci-n-multiclase)" utilizando la biblioteca Pytorch. El ejercicio se basa en el conjunto de datos de la marca de vino[Vinho verde](https://www.kaggle.com/datasets/yasserh/wine-quality-dataset).

# Importando dependencias

In [1]:
import torch
import warnings
import numpy

# Para obtener resultados reproducibles
numpy.random.seed(123)
torch.manual_seed(123)

warnings.filterwarnings("ignore")

# Funciones auxiliares
## Early stopping

In [2]:
class EarlyStopping:
    def __init__(self, paciencia: int = 5, delta: float = 0.5, modo: str = "min"):
        """
        Inicializa el callback y establece los hiperparámetros que seguirá el modelo.
        
        Args:
            paciencia (int): Número de iteraciones que el callback esperará cambios en la métrica antes de detener el entrenamiento.
            delta (float): Umbral mínimo que se espera que la métrica cambie.
            modo (str): Modo en que se evaluará la métrica, 'min' para verificar que la métrica haya disminuido (usar con el error)
            o 'max' para ver si ha aumentado (usar con accuracy, auc, f1 score, etc.).
        """
        # Cuenta la cantidad de iteraciones en las que la métrica proveída no ha superado el umbral de cambio.
        self.contador = 0
        # Controla la detención del modelo.
        self.detener = False

        # Almacena el mejor estado del modelo.
        self.mejor_estado = None
        # Almacena la iteración (número) en que se ha obtenido el mejor valor de la métrica
        self.mejor_iter = 1
        # Almacena el mejor valor de la métrica que se provea.
        self.mejor_valor = None

        # Parámetros del modelo
        self.paciencia = paciencia
        self.delta = delta
        # Modo en que se evalua la mejora de la métrica
        self.modo = modo

    def hay_mejora(self, metrica: float):
        """
        Determina si ha habido una mejora significativa en la métrica.
        
        Args:
            metrica (float): Valor de la métrica o la función de error en la iteración actual.

        Returns:
            bool: Indicador de si se ha superado el umbral de mejora entre el valor actual de la métrica y el valor registrado
            más alto.
        """
        if self.mejor_valor is None:
            return True

        if self.modo == "max":
            return metrica > (self.mejor_valor + self.delta)
        else:
            return metrica < (self.mejor_valor - self.delta)

    def evaluar_detencion(self, metrica: float, modelo, iteracion: int):
        """
        Evalua si es necesario detener el entrenamiento del modelo si no hay cambios significativos en la métrica proveída.

        Args:
            metrica (float): Valor de la métrica o la función de error en la iteración actual.
            modelo (torch.nn.Module): Modelo de red neuronal que se está entrenando.
            iteración (int): Iteración actual.
        """
        # Evalua si el cambios en la métrica entre la iteración actual y el mejor valor registrado no supera el umbral.
        if self.hay_mejora(metrica):
            self.mejor_valor = metrica
            self.mejor_estado = modelo.state_dict()
            self.mejor_iter = iteracion
            self.contador = 0
        else:
            self.contador += 1
            if self.contador >= self.paciencia:
                self.detener = True
            
    def cargar_mejor_modelo(self, modelo):
        """
        Carga el mejor estado registrado en el modelo.

        Args:
            modelo (torch.nn.Module): Modelo de red neuronal que se está entrenando.
        """
        modelo.load_state_dict(self.mejor_estado)

    def ver_mejor_iter(self):
        """
        Retorna la iteración en la que se obtuvo el mejor estado (mejor métrica).

        Returns:
            int: Iteración en la que se obtuvo la mejor métrica.
        """
        return self.mejor_iter

## Regularización L1, L2 y ElasticNet

In [3]:
def regularizacion(parametros: list[Tensor], tipo: int = 1, coef_1: float = 0.1, coef_2: float = 0.1) -> float:
    """
    Aplica regularización L1 o L2 (weight decay) a los parámetros del modelo.

    Args:
        parametros (list[Tensor]): Parámetros del modelo (matriz de pesos).
        tipo (int[1|2|3]): Tipo de regularización a utilizar. Para L1 utilice el valor 1,  para L2 el valor 2 y para ElasticNet
        utilice el valor 3.
        coef_1 (float): Coeficiente de regularización. Si está utilizando ElasticNet, se aplicará al factor L1. En caso contrario 
        es el coeficiente que se utilizará para la regularización L1 o L2 (según lo que haya escogido).
        coef_2 (float): Coeficiente de regularización para L2. Este parámetro solo es requerido si desea utilizar ElasticNet.

    Returns:
       float: Valor del coeficiente de regularización.
    """
    match (tipo):
        case "l1":
            return coef_1*sum(i.abs().sum() for i in parametros)
        case "l2":
            return coef_1*sum(i.pow(2).sum() for i in parametros)
        case "elasticnet":
            return (coef_1*sum(i.abs().sum() for i in parametros)) + (coef_2*sum(i.pow(2).sum() for i in parametros))
        case _:
            raise ValueError("Valor del parámetro 'tipo' inválido.")

## Función de entrenamiento

In [34]:
def entrenar(
    dataloader_entrenamiento: DataLoader, modelo, funcion_perdida, optimizador, iteraciones: int = 5,
    regular: bool = False, coef_l1_reg: float = 0.3, earlystop: bool = False, paciencia: int = 5,
    delta: float = 0.1, dataloader_validacion: DataLoader = None, metrica: str = "error", verbose: bool = False,
    retornar_hist_error: bool = False
) -> tuple[list[dict], list[dict]] | None:
    """
    Entrena el modelo utilizando el conjunto de datos de entrenamiento durante la cantidad de iteraciones que se establezca. 
    También permite activar las funciones de regularización y early stopping durante el entrenamiento.

    Args:
        dataloader_entrenamiento (DataLoader): Dataloader que contiene la instancia del conjunto de entrenamiento.
        
        modelo (torch.nn.Module): Modelo de red neuronal a entrenar.
        
        funcion_perdida (torch.nn.Loss): Función de pérdida (no debe calcular logits).
        
        optimizador (torch.optim): Optimizador del modelo.
        iteraciones (int): Cantidad de iteraciones a entrenar el modelo. El valor predeterminado es 5.
        
        regular (bool): Activa o desactiva la regularización L1 durante el entrenamiento. De forma 
        predeterminada está desactivada (valor False).
        
        coef_l1_reg (float): Valor del coeficiente de regularización L1.

        earlystop (bool): Activa o desactiva el early stopping durante el entrenamiento. Valor True para activarlo. De forma 
        predeterminada está desactivado (valor False).

        paciencia (int): Cantidad de iteraciones que el early stopping aguardará por cambios en la métrica establecida. De forma 
        predeterminada el valor es 5.

        delta (float): Umbral mínimo que se espera que la métrica seleccionada cambie (se usa en el early stopping).

        dataloader_validacion (DataLoader): Dataloader del conjunto de validación. Obligatorio si se activa el early stopping.

        metrica (str): Métrica con la que se evaluará el earlystopping. Los valores posibles son "error" (magnitud de la función de pérdida), 
        "accuracy", "auc", "recall", "specificity", "precision".

        verbose (bool): Indicador para activar que se muestren mensajes con las métricas que obtiene el modelo al ser evaluado con el 
        conjunto de validación.
        
        retornar_hist_error (bool): Indicador para activar que se retorne el historial de error durante el entrenamiento y el historial
        de métricas si se ha proveído un conjunto de validación.

    Returns:
        tuple[list[dict]], list[dict]] | None: Si el argumento 'retornar_hist_error' es True, retorna una tupla con el
        historial de errores por cada iteración como primer elemento y un lista con el historial de métricas obtenidas en el conjunto
        de validación. En caso contrario no retorna nada.
    """
    TAM = len(dataloader_entrenamiento.dataset)
    NUM_LOTES = len(dataloader_entrenamiento)
    early_stopping = EarlyStopping(paciencia, delta, "max" if metrica != "error" else "min") if earlystop else None

    if not isinstance(iteraciones, int):
        raise ValueError("La cantidad de iteraciones debe ser un número entero.")
    
    if earlystop and dataloader_validacion is None:
        raise ValueError("Debes proporcionar el parámetro 'dataloader_validacion' si activas early stopping")

    if metrica not in ("error", "auc", "accuracy", "recall", "precision", "specificity", "f1_score"):
        raise ValueError(f"La métrica '{metrica}' no está entre las opciones disponibles.")

    DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modelo =  modelo.to(DISPOSITIVO)
    hist_error, hist_validacion = [], []
    
    for i in range(iteraciones):
        print(f"Iteración {i+1}\n---------------------------------------")
        # Se activa el modo de entrenamiento en el modelo para activar capas de dropout, etc.
        modelo.train()
        acum = 0
        acum_error = 0
        # Recorriendo los lotes del conjunto de entrenamiento
        for X, y in dataloader_entrenamiento:
            X = X.to(DISPOSITIVO)
            Y = y.to(DISPOSITIVO)
            # Realizando una inferencia
            y_gorro = modelo(X)
            # Si se activa la regularización, se calcula el coeficiente L1 y se suma a la función de pérdida
            coef_regul = regularizacion(modelo.parameters(), "l1", coef_l1_reg) if regular else 0
            # Calculo del error
            error = funcion_perdida(y_gorro, y) + coef_regul
    
            # Retropropagación
            # Calculo de los gradientes
            error.backward()
            # Se actualizan los pesos del modelo
            optimizador.step()
            # Se reinician los gradientes en 0
            optimizador.zero_grad()
    
            acum += len(X)
            perdida = error.item()
            acum_error += perdida

            if verbose:
                print(f"Error: {perdida:>5f}  [{acum:>5d}/{TAM:>5d}]")

        hist_error.append({"iteracion": i+1, "error": acum_error / NUM_LOTES})

        # Si se activa el Early stopping, se evalua el rendimiento a partir de la métrica especificada utilizando
        # el conjunto de validación proveído.
        if earlystop:
            # Calculo de las métricas sobre el conjunto de validación
            metricas = probar(dataloader_validacion, modelo, funcion_perdida, verbose)
            metricas["iteracion"] = i + 1
            hist_validacion.append(metricas)
            # Evaluación del early stopping
            early_stopping.evaluar_detencion(metricas[metrica], modelo, i)

            if early_stopping.detener:
                early_stopping.cargar_mejor_modelo(modelo)
                print(f"En la iteración {early_stopping.ver_mejor_iter() + 1} se obtuvo el valor {early_stopping.mejor_valor:0.2f} en la métrica '{metrica}'.")
                break
        print("---------------------------------------")
    print("\n¡Entrenamiento terminado!")

    return (hist_error, hist_validacion) if retornar_hist_error else None

## Función de prueba

In [26]:
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score
from imblearn.metrics import specificity_score

def probar(dataloader: DataLoader, modelo, funcion_perdida, verbose: bool = True):
    """
    Realiza inferencias al modelo sobre el conjunto de prueba que se provea, también calcula las métricas del mismo.

    Args:
        dataloader (DataLoader): Dataloader del conjunto de prueba.
        modelo (torch.nn.Module): Modelo de red neuronal a evaluar.
        funcion_perdida (torch.nn.Loss): Función de pérdida (debe calcular logits).
        verbose (bool): Indicador para activar que se muestren mensajes con las métricas que obtiene el modelo al ser evaluado.

    Returns:
        dict: Métricas del modelo sobre el conjunto de datos (auc, error promedio, accuracy, recall, precision y specificity).
    """
    NUM_LOTES = len(dataloader)
    DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Inicialización de métricas.
    error = 0
    Y, y_pred, probs = [], [], []
    
    # Coloca el modelo en modo de inferencia, por lo que desactiva capas como dropout y normalización
    modelo.eval()

    # Se desactiva el calculo de los gradientes sobre los tensores del modelo
    with torch.no_grad():
        # Recorriendo cada lote del conjunto de prueba
        for X, y in dataloader:
            X = X.to(DISPOSITIVO)
            y = y.to(DISPOSITIVO)
            # Inferencia del modelo sobre el lote de datos.
            logits = modelo(X)
            error += funcion_perdida(logits, y).item()

            # Codificación de la salida a las etiquetas
            probabilidades = logits.softmax(1)
            y_gorro = logits.argmax(1)

            # Concatenando las salidas, inferencias del modelo y probabilidades
            Y.extend(y.cpu().numpy())
            y_pred.extend(y_gorro.cpu().numpy())
            probs.extend(probabilidades.cpu().numpy())

    # Error promedio
    error /= NUM_LOTES

    TIPO_MEDIA = "weighted"

    # Calculo de métricas
    accuracy = accuracy_score(Y, y_pred)*100
    recall = recall_score(Y, y_pred, average=TIPO_MEDIA)*100
    precision = precision_score(Y, y_pred, average=TIPO_MEDIA)*100
    f1_score_ = f1_score(Y, y_pred, average=TIPO_MEDIA)*100
    specificity = specificity_score(Y, y_pred, average=TIPO_MEDIA)*100

    try:
        auc = roc_auc_score(Y, probs, multi_class="ovo", average=TIPO_MEDIA)*100
    except ValueError:
        auc = float("nan")

    if verbose:
        print(f"\nError promedio: {error:>5f}, Accuracy: {accuracy:0.2f}%, Recall: {recall:0.2f}%, Precision: {precision:0.2f}%, F1-score: {f1_score_:0.2f}%, Specificity: {specificity:0.2f}%, AUC ROC: {auc:0.2f}%")

    return {"accuracy": accuracy, "recall": recall, "precision": precision, "specificity": specificity, "f1_score": f1_score_, "auc": auc, "error": error}

# Carga de datos
Se cargan los datos del archivo y se crean las particiones de entrenamiento y prueba. Dado que Pytorch trabaja con tensores, es necesario convertir el atributo de clase como un arreglo de Numpy. También se elimina el atributo "Id" puesto que es irrelevante y se cambia el nombre de las etiquetas para que inicie desde el valor 0.

In [6]:
from pandas import read_csv
from sklearn.model_selection import train_test_split

RUTA = "/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv"

# Carga de datos y cambio en el atributo de clase
datos = read_csv(RUTA)
datos.drop(columns=["Id"], inplace=True)
datos["quality"] -= 3

# Partición de datos
X_train, X_test, y_train, y_test = train_test_split(datos.drop(columns=["quality"]), datos["quality"],
                                                    test_size=0.2, train_size=0.8, random_state=123)
y_train, y_test = y_train.to_numpy(), y_test.to_numpy()

## Preprocesamiento de datos
Se repite el mismo procedimiento que en la versión de scikit (selección de atributos más relevantes, imputación y normalización de valores).

In [7]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PIPELINE = Pipeline([
    # Seleccionando los atributos más relevantes
    ("seleccion_atributos", SelectKBest(f_classif, k=5)),
    # En caso de haber valores nulos en alguna columna, se rellena con la mediana
    ("rellenado", SimpleImputer(strategy="median")),
    # Se normalizan los valores de cada columna
    ("normalizacion", StandardScaler())
])

x_train, x_test = PIPELINE.fit_transform(X_train, y_train), PIPELINE.transform(X_test)

# Definición del cojunto de datos
Luego de preprocesar los datos, estos son exportados como arreglos de Numpy por parte del pipeline, por lo que no es necesario convertirlos a este formato. Solo resta convertir cada instancia en un tensor, instanciar la clase para ambas particiones e instanciar los dataloader.

In [8]:
from numpy import ndarray
from torch import Tensor
from torch.utils.data import Dataset, DataLoader

class DatasetVino (Dataset):
    def __init__(self, datos: ndarray, etiquetas: ndarray):
        """
        Inicializa el conjunto de datos a partir de las instancias que se provean. Separa la etiqueta de clase de 
        los demás atributos.

        Args:
            datos (ndarray): Instancias del conjunto de datos, no contiene el atributo de clase.
            etiquetas (ndarray): Etiquetas del conjunto de datos.
        """
        self.conjunto_datos = datos.copy()
        self.etiquetas = etiquetas.copy()
        
    def __len__(self) -> int:
        """
        Retorna la cantidad de instancias presentes en el conjunto de datos.
        
        Returns:
            int: Cantidad de instancias dentro del conjunto de datos
        """
        return len(self.conjunto_datos)

    def __getitem__(self, idx: int) -> tuple[ndarray, int]:
        """
        Retorna una instancia del conjunto de datos a partir de su índice.
        
        Args:
            idx (int): Índice de la instancia.

        Returns:
            tuple[ndarray, int]: Una tupla donde el primer elemento son los atributos de la instancia y el
            segundo elemento la etiqueta de clase.
        """
        x = torch.tensor(self.conjunto_datos[idx], dtype=torch.float32)
        y = torch.tensor(self.etiquetas[idx])
        return x, y

datos_entrenamiento = DatasetVino(x_train, y_train)
datos_prueba = DatasetVino(x_test, y_test)

# 128 instancias por lote
BATCH_SIZE = 128

dataloader_entrenamiento = DataLoader(datos_entrenamiento, batch_size=BATCH_SIZE)
dataloader_prueba = DataLoader(datos_prueba, batch_size=BATCH_SIZE)

# Modelo
Se define el modelo de red neuronal que obtuvo las mejores métricas en Scikit, es decir, aquel que utilizaba solo 5 atributos sin utilizar SMOTE para balancear las clases.

**Topología:**
- **Capa de entrada:** 5 neuronas.
- **Capas ocultas:**
    - **1° capa:** 5 neuronas.
    - **2° capa:** 5 neuronas.
    - **3° capa:** 5 neuronas.
    - **4° capa:** 7 neuronas.
    - **5° capa:** 5 neuronas.
- **Capa salida** 6 neuronas.

La función de activación es **reLU** y **softmax** para la capa de salida. También se añadió inicialización de pesos.

In [35]:
from torch import nn

class PerceptronMulticapa(nn.Module):
    def __init__(self):
        """
        Constructor que instancia las capas del modelo e inicializar los pesos de las mismas.
        """
        super(PerceptronMulticapa, self).__init__()
        self.capas = nn.Sequential(
            # Capa entrada
            nn.Linear(5, 5),
            nn.ReLU(),
            # Capas ocultas
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 7),
            nn.ReLU(),
            nn.Linear(7, 5),
            nn.ReLU(),
            # Capa salida (softmax)
            nn.Linear(5, 6)
        )

        self._inicializar_pesos()

    def _inicializar_pesos(self):
        """
        Inicializa los pesos de la red utilizando el algoritmo de Glorot-Xavier. Al mismo tiempo,
        establece que el valor de los sesgos en cada neurona será 0.
        """
        for nombre, parametro in self.capas.named_parameters():
            if ("weight" in nombre) and (parametro.dim() >= 2):
                nn.init.xavier_normal_(parametro)
            elif "bias" in nombre:
                nn.init.constant_(parametro, 0.0)

    def forward(self, x: int):
        """
        Paso hacia delante, evalua una instancia (o conjunto de instancias) por la red, sin calcular
        la función de activación en la última capa.

        Args:
            x (Tensor): Instancias a evaluar por la red.

        Returns:
            Tensor: Logits de las instancias ingresadas (no son probabilidades). Deben pasarse por la función softmax.
        """
        return self.capas(x)

MODELO = PerceptronMulticapa()
OPTIMIZADOR = torch.optim.Adam(MODELO.parameters(), lr=1e-3)#, weight_decay=0.3) # Para activar la regularización L2 
FUNCION_ERROR = nn.CrossEntropyLoss() 

## Entrenamiento del modelo

In [36]:
ITERS = 200

# Usamos el Accuracy como métrica para activar el early stopping
entrenar(
    dataloader_entrenamiento, MODELO, FUNCION_ERROR, OPTIMIZADOR, ITERS, False, 0.5, True, 20, 0.1, dataloader_prueba, "accuracy", True, False
)

Iteración 1
---------------------------------------
Error: 1.771994  [  128/  914]
Error: 1.756411  [  256/  914]
Error: 1.736804  [  384/  914]
Error: 1.771693  [  512/  914]
Error: 1.762333  [  640/  914]
Error: 1.773548  [  768/  914]
Error: 1.759730  [  896/  914]
Error: 1.739157  [  914/  914]

Error promedio: 1.744349, Accuracy: 22.71%, Recall: 22.71%, Precision: 22.55%, F1-score: 20.30%, Specificity: 79.93%, AUC ROC: 52.02%
---------------------------------------
Iteración 2
---------------------------------------
Error: 1.728227  [  128/  914]
Error: 1.732780  [  256/  914]
Error: 1.689660  [  384/  914]
Error: 1.749348  [  512/  914]
Error: 1.741024  [  640/  914]
Error: 1.752749  [  768/  914]
Error: 1.722506  [  896/  914]
Error: 1.741396  [  914/  914]

Error promedio: 1.717411, Accuracy: 23.58%, Recall: 23.58%, Precision: 22.07%, F1-score: 20.88%, Specificity: 78.49%, AUC ROC: 53.55%
---------------------------------------
Iteración 3
--------------------------------------